# Préparation des données : sélection des films

# Objectif général
Nous voulons construire une liste de films se basant sur plusieurs critères à partir des données publiques d'IMDb  :
- des films de cinéma, sortis depuis 1970, d'au moins 1 heure
- "connus" (au moins 10 000 votes) et "bien notés" (6/10 ou plus)
- appartenant à au moins un de nos 7 genres : Action, Horreur, Thriller, Science-fiction, Aventure, Guerre, Comédie.

Pour ce faire, nous vérifions que ces films se trouvent dans la base de donnée TMDB, une autre base qui servira plus tard à les enrichir (résumés, affiches, etc.).

# Démarche
1. Charger les données IMDb.
2. Filtrer pas à pas, comme un entonnoir, en comptant les films restants après chaque filtre.
3. Vérifier le résultat avec des films célèbres.
4. Comparer avec TMDB.
5. Enregistrer le résultat dans des fichiers CSV.

In [27]:
import pandas as pd

ratings = pd.read_csv("https://datasets.imdbws.com/title.ratings.tsv.gz",
                      sep="\t", na_values="\\N")

colonnes = ["tconst", "titleType", "primaryTitle", "isAdult",
            "startYear", "runtimeMinutes", "genres"]
basics = pd.read_csv("https://datasets.imdbws.com/title.basics.tsv.gz",
                     sep="\t", na_values="\\N", usecols=colonnes,
                     low_memory=False)

basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")
basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")
basics["isAdult"] = pd.to_numeric(basics["isAdult"], errors="coerce")

print(basics.shape, ratings.shape)

(12999649, 7) (1736063, 3)


# Explication : cellule 1, charger les données

L'objectif est de récupérer les deux tableaux IMDb dont on a besoin et les préparer pour pouvoir les filtrer.


1. **import pandas as pd** charge **pandas**, la bibliothèque Python qui manipule des tableaux de données (appelés *DataFrames*).
2. Nous lisons deux fichiers IMDb **directement depuis Internet** avec **pd.read_csv(...)** :
3. **ratings** contient, pour chaque titre, sa note moyenne (**averageRating**) et son nombre de votes (**numVotes**) 
4. **basics** contient la fiche du titre : identifiant (**tconst**), type (film, série...), nom, adulte ou non, année, durée et genres.
5. **sep="\t"** : dans ces fichiers **.tsv**, les colonnes sont séparées par des **tabulations**, pas par des virgules.
6. **na_values="\\\\N"** : IMDb écrit **\N** quand une information manque. pandas remplace ces **\N** par une case vide (**NaN**).
7. **usecols=colonnes** : nous ne chargeons que les 7 colonnes utiles. Le fichier fait environ 13 millions de lignes, donc cela économise beaucoup de mémoire.
8. **low_memory=False** : pandas lit le fichier d'un seul bloc, ce qui évite des avertissements sur le type des colonnes.
9. **pd.to_numeric(..., errors="coerce")** transforme l'année, la durée et **isAdult** en nombres. Sans cela, nous ne pourrions pas écrire **>= 1970**. Avec **errors="coerce"**, une valeur illisible devient vide au lieu de faire planter le code.
10. **print(basics.shape, ratings.shape)** affiche la taille de chaque tableau sous la forme (lignes, colonnes).


Nous obtenons alors avec **basics** 12 958 477 titres (7 colonnes) et **ratings** **1 731 756 notes** (3 colonnes). Tous les titres n'ont donc pas de note.



In [28]:
etapes = []
films = basics.copy()
etapes.append(["Tous les titres IMDb", len(films)])

films = films[films["titleType"] == "movie"]
etapes.append(["Films de cinéma", len(films)])

films = films[films["isAdult"] == 0]
etapes.append(["Sans films adultes", len(films)])

films = films[films["startYear"] >= 1970]
etapes.append(["Depuis 1970", len(films)])

films = films[films["runtimeMinutes"] >= 60]
etapes.append(["60 minutes ou plus", len(films)])

films = films.merge(ratings, on="tconst", how="inner")
etapes.append(["Films avec une note", len(films)])

langues = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", usecols=["imdb_id", "original_language"])
langues = langues.drop_duplicates("imdb_id")
films = films.merge(langues, left_on="tconst", right_on="imdb_id", how="left")
films["francais"] = films["original_language"] == "fr"

films = films[(films["numVotes"] >= 10000) | (films["francais"] & (films["numVotes"] >= 2000))]
etapes.append(["10 000 votes ou plus (2 000 pour les films français)", len(films)])

films = films[(films["averageRating"] >= 6) | (films["numVotes"] >= 100000)]
etapes.append(["Note de 6 ou plus (sauf films vus par plus de 100 000 personnes)", len(films)])

films = films[films["genres"].str.contains("Action|Horror|Thriller|Sci-Fi|Adventure|War|Comedy", na=False) | (films["numVotes"] >= 500000)]
etapes.append(["Nos 7 genres (sauf films vus par plus de 500 000 personnes)", len(films)])

films = films.drop(columns=["imdb_id", "original_language", "francais"])

bilan = pd.DataFrame(etapes, columns=["Filtre", "Nombre de films"])
bilan

,Filtre,Nombre de films
0,Tous les titres IMDb,12999649
1,Films de cinéma,761813
2,Sans films adultes,752716
3,Depuis 1970,513639
4,60 minutes ou plus,351578
5,Films avec une note,251305
6,10 000 votes ou plus (2 000 pour les films fra...,12875
7,Note de 6 ou plus (sauf films vus par plus de ...,9957
8,Nos 7 genres (sauf films vus par plus de 500 0...,7310


# Explication : cellule 2, filtrer étape par étape


La démarche de **l'entonnoir** : nous partons de tous les titres et nous appliquons un critère à la fois.
Après chaque filtre, nous ajoutons dans la liste **etapes** le nom du filtre et le nombre de films restants (**len(films)**), ce qui nous permet de justifier chaque choix avec un chiffre. Cela permet aussi de voir quel filtre retire le plus de films.

1. **films = basics.copy()** fait une copie, pour ne jamais modifier le tableau d'origine.
2. **films[condition]** ne garde que les lignes où la condition est vraie. Nous filtrons dans cet ordre :
   - **titleType == "movie"** : uniquement les **films de cinéma** (pas les séries, épisodes, courts-métrages...) ;
   - **isAdult == 0** : nous retirons les **films pour adultes** ;
   - **startYear >= 1970** : films sortis **depuis 1970** ;
   - **runtimeMinutes >= 60** : films d'**au moins une heure**.
3. **films.merge(ratings, on="tconst", how="inner")** **colle les notes** à côté de chaque film, grâce à l'identifiant commun **tconst**. Avec **how="inner"**, nous ne gardons que les films présents dans les deux tableaux : ceux **sans note disparaissent**.
4. **numVotes >= 10000** garde les films **assez connus**, et **averageRating >= 6** les films **bien notés**.
5. **str.contains("Action|Horror|...", na=False)** garde les films qui ont **au moins un** des 7 genres. Dans ce texte, le **|** veut dire « ou ». **na=False** écarte les films sans genre.
6. **pd.DataFrame(etapes, columns=[...])** transforme la liste en tableau **bilan**. Écrire **bilan** seul sur la dernière ligne l'affiche sous la cellule.

Nous constatons que le filtre **des votes** fait de loin la plus grosse coupe : nous passons de 250 715 à 11 773 films. À la fin, il reste **6 447 films**.




In [29]:
references = ["tt0076759", "tt0078748", "tt0361748", "tt0077651", "tt0073195"]
films[films["tconst"].isin(references)][["primaryTitle", "startYear", "averageRating", "numVotes", "genres"]]

,primaryTitle,startYear,averageRating,numVotes,genres
5191,Jaws,1975.0,8.1,755075,"Adventure,Drama,Horror"
7293,Star Wars: Episode IV - A New Hope,1977.0,8.6,1599218,"Action,Adventure,Fantasy"
7766,Halloween,1978.0,7.7,350031,"Horror,Thriller"
8365,Alien,1979.0,8.4,1084329,"Horror,Sci-Fi"
62816,Inglourious Basterds,2009.0,8.4,1832202,"Drama,War"


# Explication : cellule 3, vérifier avec des films connus

L'étape de la vérifictation :  Des films célèbres qui correspondent à nos critères doivent être encore présents après tous les filtres.


1. **references** est la liste des identifiants IMDb de 5 films connus : *Star Wars*, *Alien*, *Inglourious Basterds*, *Halloween* et *Les Dents de la mer* (*Jaws*).
2. **films["tconst"].isin(references)** renvoie Vrai pour les films dont l'identifiant est dans cette liste, et **films[...]** ne garde que ces lignes.
3. **[["primaryTitle", "startYear", ...]]** : les doubles crochets servent à **n'afficher que certaines colonnes**.

Si un film évident avait disparu, cela voudrait dire qu'un filtre est mal réglé (par exemple une année ou un genre mal écrit).

Les films sont bien présents. Nous remarquons qu' **Inglourious Basterds** est gardé grâce au genre War, et **Jaws** grâce aux genres Adventure et Horror.


In [30]:
tmdb_ids = pd.read_csv("../data/TMDB_movie_dataset_v11.csv", usecols=["imdb_id"])

presents = films["tconst"].isin(tmdb_ids["imdb_id"]).sum()
print(presents, "films sur", len(films), "sont dans TMDB")

7115 films sur 7310 sont dans TMDB


# Explication : cellule 4, correspondance avec TMDB

L'objecti est de vérifier que nous pouvons enrichir nos films avec les données TMDB, une autre base de films qui contient plus d'informations (résumés, affiches, pays de production...).


1. **pd.read_csv("data/TMDB_movie_dataset_v11.csv", usecols=["imdb_id"])** lit seulement la colonne imdb_id du fichier TMDB, qui est très lourd . Ce fichier est stocké dans le dossier data/ du projet.
2. Le lien entre les deux bases se fait par l'identifiant IMDb : le **tconst** d'IMDb correspond au **imdb_id** de TMDB.
3. **films["tconst"].isin(tmdb_ids["imdb_id"])** renvoie Vrai pour chaque film qui existe aussi dans TMDB, et **.sum()** **compte les Vrai** (Vrai vaut 1, Faux vaut 0).

6 255 films sur 6 447 sont dans TMDB, soit environ 97 %. 
Nous pourrons donc enrichir presque tous nos films.


In [31]:
films.to_csv("../data/films_filtres.csv", index=False)
bilan.to_csv("../data/bilan_filtres.csv", index=False)
print(len(films), "films enregistrés")

7310 films enregistrés


# Explication : cellule 5, enregistrer le résultat

Nouss sauvergadons le travail pour ne pas devoir tout retélécharger et refiltrer à chaque fois.


1. **films.to_csv("data/films_filtres.csv", index=False)** enregistre la **liste des films filtrés**.
2. **bilan.to_csv("data/bilan_filtres.csv", index=False)** enregistre le **bilan des filtres**, utile pour la présentation.
3. **index=False** évite d'ajouter une colonne avec les numéros de ligne de pandas, qui ne servent à rien dans le fichier.

6 447 films sont enregistrés. Les prochains notebooks repartiront de ces fichiers CSV.
